# 05 Batch Diagnostics

**Purpose.** Patient-identifier baseline, global acquisition properties, acquisition-order matching.

**Thesis reference.** §4.6 Acquisition-Order Confounding

**Inputs / outputs.** See the data-layout table in the repository README. Paths near the top of the notebook (e.g. `PROFILE_DIR`, `IMAGE_ROOT`, `LABELS`) are relative to the repository root and assume the folder layout described there.

---

In [26]:
PROFILE_DIR         = "profiles"
LABELS_FILE         = "labels/labels_detect_nondiabetic.json"
OUT_FILE            = "batch_check_nondiabetic.json"
MATCHED_LABELS_FILE = "labels/labels_matched_nondiabetic.json"
MATCH_TOLERANCE     = 60

## Code

In [ ]:
import json
from pathlib import Path

import numpy as np
from scipy.stats import mannwhitneyu, spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def pid_num(pid):
    return int("".join(ch for ch in pid if ch.isdigit()))

def main():
    prof = Path(PROFILE_DIR)
    F = np.load(prof / "mz_profiles.npy")
    with open(prof / "profile_pids.json") as fh:
        pids = json.load(fh)
    with open(LABELS_FILE) as fh:
        labels = json.load(fh).get("labels")

    keep = [i for i, p in enumerate(pids) if p in labels]
    F = F[keep]
    kept = [pids[i] for i in keep]
    y = np.array([labels[p] for p in kept])
    ids = np.array([pid_num(p) for p in kept])

    print(f"Patients: {len(y)}  ({int(y.sum())} CKD+ / {int((1-y).sum())} CKD-)\n")

    # global spectral properties, per patient
    total_int = F.sum(axis=1)
    n_occupied = (F > 0).sum(axis=1)
    med_int = np.median(F, axis=1)
    p95 = np.percentile(F, 95, axis=1)
    # centre of mass along the m/z axis: shifts if calibration drifts
    w = F / F.sum(axis=1, keepdims=True).clip(min=1e-12)
    com = (w * np.arange(F.shape[1])).sum(axis=1)

    props = {
        "total_intensity": total_int,
        "occupied_bins": n_occupied,
        "median_intensity": med_int,
        "p95_intensity": p95,
        "mz_centre_of_mass": com,
    }

    print("1. Does each property trend with patient ID?")
    print(f"{'property':22s} {'rho':>8} {'p':>12}")
    trend = {}
    np.save("drift_ids.npy", ids)
    for k, v in props.items():
        rho, p = spearmanr(ids, v)
        trend[k] = {"rho": float(rho), "p": float(p)}
        if k == "mz_centre_of_mass":         
            np.save("drift_com.npy", v)
        flag = "  <-- drift" if p < 0.01 and abs(rho) > 0.2 else ""
        print(f"{k:22s} {rho:8.3f} {p:12.2e}{flag}")

    print("\n2. Does each property differ between classes?")
    print(f"{'property':22s} {'AUC':>8} {'p':>12}")
    byclass = {}
    for k, v in props.items():
        u, p = mannwhitneyu(v[y == 1], v[y == 0])
        auc = u / ((y == 1).sum() * (y == 0).sum())
        byclass[k] = {"auc": float(max(auc, 1 - auc)), "p": float(p)}
        flag = "  <-- differs" if p < 0.01 else ""
        print(f"{k:22s} {max(auc,1-auc):8.3f} {p:12.2e}{flag}")

    print("\n3. AUC from global properties alone (5-fold CV)")
    X = np.column_stack(list(props.values()))
    clf = make_pipeline(StandardScaler(),
                        LogisticRegression(max_iter=5000,
                                           class_weight="balanced"))
    pr = cross_val_predict(clf, X, y, cv=5, method="predict_proba")[:, 1]
    auc_props = roc_auc_score(y, pr)
    u, _ = mannwhitneyu(ids[y == 1], ids[y == 0])
    auc_id = u / ((y == 1).sum() * (y == 0).sum())
    auc_id = max(auc_id, 1 - auc_id)
    print(f"  patient ID alone      : {auc_id:.4f}")
    print(f"  global properties     : {auc_props:.4f}")
    print("\n  Any model must be compared against these. A spectral model that")
    print("  does not clearly exceed them has not demonstrated biological signal.")

    # 4. ID-matched subcohort
    print("\n4. Building an ID-matched subcohort")
    cases = sorted(np.where(y == 1)[0], key=lambda i: ids[i])
    ctrls = sorted(np.where(y == 0)[0], key=lambda i: ids[i])
    used, pairs = set(), []
    for c in ctrls:
        best, bestd = None, MATCH_TOLERANCE + 1
        for k in cases:
            if k in used:
                continue
            d = abs(int(ids[k]) - int(ids[c]))
            if d < bestd:
                best, bestd = k, d
        if best is not None:
            used.add(best)
            pairs.append((c, best))

    if pairs:
        idx = np.array([i for pr_ in pairs for i in pr_])
        ym, idm = y[idx], ids[idx]
        u, p = mannwhitneyu(idm[ym == 1], idm[ym == 0])
        auc_m = u / ((ym == 1).sum() * (ym == 0).sum())
        print(f"  matched pairs: {len(pairs)}  (n = {len(idx)})")
        print(f"  patient ID AUC within matched cohort: "
              f"{max(auc_m, 1-auc_m):.4f}  (p = {p:.3f})")
        matched = [kept[i] for i in idx]
        Path(MATCHED_LABELS_FILE).parent.mkdir(parents=True, exist_ok=True)
        with open(MATCHED_LABELS_FILE, "w") as fh:
            json.dump({"task": "id_matched",
                       "note": "ID-matched subcohort; acquisition order "
                               "cannot discriminate",
                       "labels": {p_: int(labels[p_]) for p_ in matched}},
                      fh, indent=1)
        print(f"  -> {MATCHED_LABELS_FILE} written")
    else:
        matched = []
        print("  no pairs found within tolerance")

    with open(OUT_FILE, "w") as fh:
        json.dump({"n": len(y), "auc_patient_id": float(auc_id),
                   "auc_global_properties": float(auc_props),
                   "id_trend": trend, "class_difference": byclass,
                   "n_matched": len(matched)}, fh, indent=2)
    print(f"\nWritten to {OUT_FILE}")

## Run

In [27]:
main()

Patients: 405  (359 CKD+ / 46 CKD-)

1. Does each property trend with patient ID?
property                    rho            p
total_intensity          -0.101     4.13e-02
occupied_bins             0.060     2.29e-01
median_intensity         -0.207     2.70e-05  <-- drift
p95_intensity            -0.067     1.77e-01
mz_centre_of_mass        -0.493     3.07e-26  <-- drift

2. Does each property differ between classes?
property                    AUC            p
total_intensity           0.588     5.17e-02
occupied_bins             0.585     5.22e-02
median_intensity          0.541     3.63e-01
p95_intensity             0.587     5.38e-02
mz_centre_of_mass         0.546     3.15e-01

3. AUC from global properties alone (5-fold CV)
  patient ID alone      : 0.6947
  global properties     : 0.5244

  Any model must be compared against these. A spectral model that
  does not clearly exceed them has not demonstrated biological signal.

4. Building an ID-matched subcohort
  matched pairs: 46

In [19]:
import json
d = json.load(open("labels/labels_matched_nondiabetic.json"))
print("top-level keys:", list(d)[:8])
lab = d.get("labels", d)
print("n entries:", len(lab))
print("first 3:", list(lab.items())[:3])

top-level keys: ['n', 'auc_patient_id', 'auc_global_properties', 'id_trend', 'class_difference', 'n_matched']
n entries: 6
first 3: [('n', 405), ('auc_patient_id', 0.6946832990190142), ('auc_global_properties', 0.5244035363933632)]
